 # Extract Chlorophyll from Landsat Using for a 500 m x 500 m Region Centered on a Lake

In [ ]:
import ee
import math
import pandas as pd
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg') # specify a registered GEE Cloud Project

    # rows = series.getInfo()['features']                      # request JSON to client
    # records = [f['properties'] for f in rows]            # strip geometry
    # df = pd.DataFrame.from_records(records)
    # df.to_csv(export_id, index=False)


## Landsat

In [ ]:
# ------------ USER LIST ----------------------------------------------------
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         export_id='Detroit_Landsat_NDCI_500m'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         export_id='Klamath_Landsat_NDCI_500m')
]

start_date, end_date = '2011-01-01', '2025-12-31'
half_size = 250     # metres (gives 500-m box)

# ------------ COLLECTIONS & MASK ------------------------------------------
landsat = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
           .merge(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2'))
           .merge(ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')))

def mask_clouds(img):
    qa = img.select('QA_PIXEL')
    mask = (qa.bitwiseAnd(1 << 3).eq(0)   # cloud
            .And(qa.bitwiseAnd(1 << 4).eq(0))   # shadow
            .And(qa.bitwiseAnd(1 << 5).eq(0))   # snow
            .And(qa.bitwiseAnd(1 << 2).eq(0)))  # cirrus
    return img.updateMask(mask)

def add_water_mask(img):
    bands = img.bandNames()

    # Choose NIR band depending on sensor
    nir = ee.Algorithms.If(
            bands.contains('SR_B5'),  # OLI/OLI-2
            'SR_B5',
            'SR_B4'                   # TM and ETM+
          )

    ndwi = img.normalizedDifference(['SR_B3', ee.String(nir)])
    return img.updateMask(ndwi.gt(0))

def drop_low_edge(img, edge_band, thresh=0.002):
    edge = img.select(edge_band).multiply(1e-4 if edge_band=='B5' else 2.75e-5).add(-0.2)
    return img.updateMask(edge.gt(thresh))

def add_ndci(img):
    sr = img.select(['SR_B4', 'SR_B5']).multiply(2.75e-5).add(-0.2)
    ndci = sr.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDCI')
    return img.addBands(ndci)

def img_to_feat(img, roi, tag):
    mean = img.select('NDCI').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=30,
        maxPixels=1e9).get('NDCI')
    return ee.Algorithms.If(
        mean,
        ee.Feature(None, {
            'date'  : ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'ndci'  : mean,
            'sensor': tag}),
        None)

def preprocess_landsat(img):
    img = mask_clouds(img)                     # your QA_PIXEL cloud mask
    img = add_water_mask(img)                  # NDWI > 0 keeps water only
    # optional: drop extremely dark red-edge pixels (deep shadow)
    edge = img.select('SR_B5').multiply(2.75e-5).add(-0.2) \
            if img.bandNames().contains('SR_B5') \
            else img.select('SR_B4').multiply(2.75e-5).add(-0.2)
    img = img.updateMask(edge.gt(0.002))
    return img

# ------------ LOOP THROUGH LAKES ------------------------------------------
for lake in lakes:
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi    = center.buffer(half_size).bounds()

    series = (landsat
              .filterDate(start_date, end_date)
              .filterBounds(roi)
              .map(preprocess_landsat)
              .map(add_ndci)
              .map(lambda img: img_to_feat(img, roi, lake['name'] + '_Landsat'),
                   dropNulls=True))

    print(lake['name'],
          'valid Landsat scenes =',
          series.aggregate_count('ndci').getInfo())

    # ee.batch.Export.table.toDrive(
    #     collection  = series,
    #     description = lake['export_id'],
    #     fileFormat  = 'CSV'
    # ).start()

    rows = series.getInfo()['features']                      # request JSON to client
    records = [f['properties'] for f in rows]            # strip geometry
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['export_id'] + '.csv', index=False)


Detroit valid Landsat scenes = 511
UpperKlamath valid Landsat scenes = 694


## Sentinel

In [20]:
# ---------------------------------------------------------------------------
# 1  LAKE LIST
# ---------------------------------------------------------------------------
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         export_id='Detroit_S2_NDCI_500m'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         export_id='Klamath_S2_NDCI_500m')
    # add more lakes here if needed
]

start_date, end_date = '2015-07-01', '2025-12-31'   # Sentinel-2 archive
half_size_m = 250                                   # 500-m square ROI

# ---------------------------------------------------------------------------
# 2  IMAGE COLLECTION AND CLOUD MASK
# ---------------------------------------------------------------------------
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

def mask_s2(img):
    qa     = img.select('QA60')
    cloud  = qa.bitwiseAnd(1 << 10).neq(0)          # opaque cloud
    cirrus = qa.bitwiseAnd(1 << 11).neq(0)          # cirrus
    mask   = cloud.Or(cirrus).Not()
    return img.updateMask(mask)\
              .updateMask(img.select('B8A').gt(0))  # remove edge-stripes

def add_water_mask_s2(img):
    ndwi = img.normalizedDifference(['B3', 'B8'])
    return img.updateMask(ndwi.gt(0))

# ---------------------------------------------------------------------------
# 3  ADD NDCI (SURFACE REFLECTANCE, NO π DIVISION)
# ---------------------------------------------------------------------------
def add_ndci(img):
    # Sentinel-2 L2A scale factor = 1e-4
    sr   = img.select(['B4', 'B5']).multiply(1e-4)
    red  = sr.select('B4')             # 665 nm
    edge = sr.select('B5')             # 705 nm (red-edge)
    ndci = edge.subtract(red)\
               .divide(edge.add(red))\
               .rename('NDCI')
    return img.addBands(ndci)

def preprocess_s2(img):
    img = mask_s2(img)                 # QA60 cloud / cirrus
    img = add_water_mask_s2(img)       # NDWI > 0 keeps water
    edge = img.select('B5').multiply(1e-4).add(0)   # B5 reflectance
    img = img.updateMask(edge.gt(0.002))
    return img

# ---------------------------------------------------------------------------
# 4  IMAGE ➜ FEATURE (MEAN NDCI INSIDE ROI)
# ---------------------------------------------------------------------------
def img_to_feature(img, roi, tag):
    mean = img.select('NDCI').reduceRegion(
        reducer   = ee.Reducer.mean(),
        geometry  = roi,
        scale     = 10,                # Sentinel-2 pixel size
        maxPixels = 1e9
    ).get('NDCI')

    return ee.Algorithms.If(
        mean,
        ee.Feature(None, {
            'date'  : ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'ndci'  : mean,
            'sensor': tag
        }),
        None)

# ---------------------------------------------------------------------------
# 5  PROCESS EACH LAKE AND EXPORT
# ---------------------------------------------------------------------------
for lake in lakes:
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi    = center.buffer(half_size_m).bounds()

    series = (s2.filterDate(start_date, end_date)
               .filterBounds(roi)
               .map(preprocess_s2)
               .map(add_ndci)
               .map(lambda img: img_to_feature(img, roi, lake['name'] + '_S2'),
                    dropNulls=True))

    print(lake['name'],
          'valid Sentinel-2 scenes =',
          series.aggregate_count('ndci').getInfo())

    # ee.batch.Export.table.toDrive(
    #     collection  = series,
    #     description = lake['export_id'],
    #     fileFormat  = 'CSV'
    # ).start()

    rows = series.getInfo()['features']                      # request JSON to client
    records = [f['properties'] for f in rows]            # strip geometry
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['export_id'] + '.csv', index=False)


Detroit valid Sentinel-2 scenes = 286
UpperKlamath valid Sentinel-2 scenes = 1556


## MODIS

In [ ]:
# ---------------------------------------------------------------------------
# Create daily MODIS Aqua and Terra chlorophyll time series for any set of
# lakes, sampling one 500-m pixel at the lake center.  Each lake receives
# two CSV exports (Aqua and Terra) written to Google Drive.
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 1  LIST OF LAKES TO PROCESS
# ---------------------------------------------------------------------------
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         terra_export='Detroit_MODIS_Terra_500m_Chl_singlePixel',
         aqua_export ='Detroit_MODIS_Aqua_500m_Chl_singlePixel'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         terra_export='Klamath_MODIS_Terra_500m_Chl_singlePixel',
         aqua_export ='Klamath_MODIS_Aqua_500m_Chl_singlePixel')
    # add more lakes here if desired
]

# ---------------------------------------------------------------------------
# 2  GLOBAL SETTINGS
# ---------------------------------------------------------------------------
start_date = '2011-01-01'
end_date   = '2025-12-31'

# Green : red algorithm coefficients
offset = 1.54          # use 0.96 for UKL low-end adjust if needed
slope  = 1.70

# ---------------------------------------------------------------------------
# 3  COMMON FUNCTIONS
# ---------------------------------------------------------------------------
def mask_light_cloud(img):
    qa = img.select('state_1km')
    cloud = qa.bitwiseAnd(1 << 10).neq(0)
    snow  = qa.bitwiseAnd(1 << 12).neq(0)
    return img.updateMask(cloud.Not()).updateMask(snow.Not())

def add_chlorophyll(img):
    sr555 = img.select('sur_refl_b04').multiply(1e-4)  # 555 nm
    sr645 = img.select('sur_refl_b01').multiply(1e-4)  # 645 nm
    rrs555 = sr555.divide(ee.Number(math.pi))
    rrs645 = sr645.divide(ee.Number(math.pi))
    log_ratio = rrs555.divide(rrs645).log10()
    log10_chl = log_ratio.multiply(slope).add(offset)
    chl = ee.Image(10).pow(log10_chl).rename('chlor_a')
    return img.addBands(chl)

def build_series(col_id, sensor_tag, pt):
    collection = (ee.ImageCollection(col_id)
                  .filterDate(start_date, end_date)
                  .filterBounds(pt)
                  .map(mask_light_cloud)
                  .map(add_chlorophyll)
                  .select('chlor_a'))

    def img_to_feature(img):
        fc = img.sample(region=pt,
                        scale=500,
                        numPixels=1,
                        geometries=False)
        return ee.Algorithms.If(
            fc.size().gt(0),
            ee.Feature(None, {
                'date'  : ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
                'chl'   : fc.first().get('chlor_a'),
                'sensor': sensor_tag
            }),
            None)

    return collection.map(img_to_feature, dropNulls=True)

# ---------------------------------------------------------------------------
# 4  LOOP THROUGH LAKES / EXPORT CSVs
# ---------------------------------------------------------------------------
for lake in lakes:
    center = ee.Geometry.Point([lake['lon'], lake['lat']])

    terra_series = build_series('MODIS/061/MOD09GA', 'Terra', center)
    aqua_series  = build_series('MODIS/061/MYD09GA', 'Aqua',  center)

    print(lake['name'], 'Terra rows =',
          terra_series.aggregate_count('chl').getInfo())
    print(lake['name'], 'Aqua  rows =',
          aqua_series.aggregate_count('chl').getInfo())

    # ee.batch.Export.table.toDrive(
    #     collection  = terra_series,
    #     description = lake['terra_export'],
    #     fileFormat  = 'CSV'
    # ).start()

    # ee.batch.Export.table.toDrive(
    #     collection  = aqua_series,
    #     description = lake['aqua_export'],
    #     fileFormat  = 'CSV'
    # ).start()

    rows = terra_series.getInfo()['features']            # request JSON to client
    records = [f['properties'] for f in rows]            # strip geometry
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['terra_export'] + '.csv', index=False)

    rows = aqua_series.getInfo()['features']             # request JSON to client
    records = [f['properties'] for f in rows]            # strip geometry
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['aqua_export'] + '.csv', index=False)


/opt/homebrew/Caskroom/miniconda/base/envs/clearwater/lib/python3.13/site-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for MODIS/006/MOD09GA! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD09GA

  warnings.warn(warning, category=DeprecationWarning)
/opt/homebrew/Caskroom/miniconda/base/envs/clearwater/lib/python3.13/site-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for MODIS/006/MYD09GA! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MYD09GA

  warnings.warn(warning, category=DeprecationWarning)


Detroit Terra rows = 1516
Detroit Aqua  rows = 1536
UpperKlamath Terra rows = 2188
